# nano-prover — fixed, RESUMABLE Kaggle runner

**One-time setup:** right panel → Accelerator **GPU T4 x2**, Internet **On**; Add-ons → Secrets → add
**`GITHUB_TOKEN`** (GitHub token, scope `repo`).

**Every run:** press ▶. The cell:
1. trains (**skipped** if a model from a previous run is already in the repo) and **pushes the model**;
2. generates eval proofs **in chunks, pushing after each** — so if the 12 h limit cuts it off, just
   **run the cell again** and it resumes from where it stopped (e.g. 88/100 → continues at 89).

Change the next run by editing `training/scripts/run_kaggle.sh` + `git push`. When all 100 are done,
locally: `git pull` (model is in `training/model/`) and `python -m assess`.

In [ ]:
# ---- FIXED + RESUMABLE: clone → (train once → push model) → chunked gen → push preds each chunk. ----
import os, subprocess
os.chdir('/kaggle/working')
subprocess.run('rm -rf Conjecture-Gym && git clone --depth 1 https://github.com/FELIXGAO0315/Conjecture-Gym.git', shell=True)
os.chdir('/kaggle/working/Conjecture-Gym')
!nvidia-smi -L
from kaggle_secrets import UserSecretsClient
TOKEN = UserSecretsClient().get_secret('GITHUB_TOKEN')
REMOTE = f'https://{TOKEN}@github.com/FELIXGAO0315/Conjecture-Gym.git'

def push(msg):
    subprocess.run('git add -f training/model training/scripts/preds.jsonl', shell=True)
    subprocess.run(f"git -c user.email=k@k.c -c user.name=k commit -m '{msg}' || true", shell=True)
    subprocess.run(f'git push {REMOTE} main', shell=True)

def n_done():
    p = 'training/scripts/preds.jsonl'
    return sum(1 for _ in open(p)) if os.path.exists(p) else 0

# 1) train ONCE — skip if the model from a previous (resumed) run is already present
if os.path.exists('training/model/adapter_config.json'):
    print('>>> model already present (resumed run) — skipping training')
else:
    subprocess.run('bash training/scripts/run_kaggle.sh train', shell=True, check=True)
    subprocess.run('rm -f training/scripts/preds.jsonl', shell=True)   # fresh eval for the new model
    push('trained adapter (post-train)')
    print('>>> MODEL SAVED and pushed — the deliverable is safe.')

# 2) resumable chunked generation — push after each chunk so a timeout loses at most one chunk
while True:
    before = n_done()
    subprocess.run('bash training/scripts/run_kaggle.sh gen', shell=True)
    push(f'preds {n_done()}/100')
    if n_done() <= before:
        break   # no new this chunk => all done (or stuck)
    print(f'--- progress: {n_done()}/100 (pushed) ---')
print(f'\nDONE. {n_done()}/100 generated. Locally:  git pull  &&  python -m assess')